# v85 submit — r0単体 と r0+affine(50:50)blend の2提出
# 目的: r0単体LB と affine混合LBの比較。struct/物理錨/dec 一切不使用。
# GRの別表現(affine較正)をPFに通して混ぜる。出力: submission_r0.csv / submission_blend.csv


In [ ]:
# ===== 設定 =====
import os, glob, time, hashlib, json
from pathlib import Path
from dataclasses import dataclass, asdict, field
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")

def _find_data():
    cands = ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
             "/kaggle/input/rogii-wellbore-geology-prediction",
             r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction"]
    for p in cands:
        if os.path.exists(os.path.join(p, "sample_submission.csv")):
            return Path(p)
    h = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    return Path(os.path.dirname(h[0])) if h else Path(".")

DATA = _find_data()
TEST = DATA / "test"
OUTDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("DATA:", DATA, "| TEST wells:", len(glob.glob(str(TEST / "*__horizontal_well.csv"))))

# r0_seed32(struct/anchor不使用の struct-free PF params)
R0 = dict(
    n_particles=400, n_seeds=32, seed0=0, likelihood_scale=10.536319667317654,
    init_pos_std=0.9331541486020234, init_rate_std=0.00047753793163756304,
    mom=0.9997911117425017, vn=0.0006297672017716754, pn=0.005014009140727263,
    rp=0.7627309890952053, rr=0.00045663430474523125, resamp=0.032049849143995945,
    gr_sig_min=11.865793233968713, gr_sig_max=44.98927964612316, gr_sig_def=119.25397464636981,
    gr_sig_mult=3.058318309943017, tvt_clip_margin=150.56111180234504, rate_clip=2.0,
    gr_power=1.7826896851395453, hgr_smooth_r=0, tw_gr_smooth_r=0, jump_prob=0.002, jump_std=5.0)
BLEND_W = 0.5     # blend = (1-W)*r0 + W*affine
SUBMIT  = "blend" # ★ submission.csv にするもの: "r0"=r0単体 / "blend"=0.5*r0+0.5*affine


In [ ]:
# ===== struct-free PF実装(pf_tuning_pf01)を埋込 =====
@dataclass
class PFParams:
    # Ensemble
    n_particles: int = 500
    n_seeds: int = 32
    seed0: int = 0
    likelihood_scale: float = 5.0

    # Initial particle distribution around last known TVT + Z
    init_pos_std: float = 4.5
    init_rate_std: float = 0.01

    # State transition noise / momentum
    mom: float = 0.998
    vn: float = 0.002
    pn: float = 0.005

    # Resampling noise / threshold
    rp: float = 0.1
    rr: float = 0.001
    resamp: float = 0.5

    # GR likelihood sigma inferred from known interval residuals
    gr_sig_min: float = 10.0
    gr_sig_max: float = 60.0
    gr_sig_def: float = 30.0
    gr_sig_mult: float = 1.0

    # TVT clipping margin outside typewell TVT range
    tvt_clip_margin: float = 100.0

    # Extra optional controls. Defaults reproduce the current/original behavior.
    # rate_clip <= 0: no clipping
    rate_clip: float = 0.0
    # Original GR likelihood uses squared normalized residuals.
    gr_power: float = 2.0
    # 0 means no GR smoothing.
    hgr_smooth_r: int = 0
    tw_gr_smooth_r: int = 0

    # Rare jump particles for extreme TVT movements.
    # jump_prob=0 reproduces the current/original behavior.
    jump_prob: float = 0.0
    jump_std: float = 10.0


def _pf_params_dict(params=None):
    if params is None:
        params = PFParams()
    if isinstance(params, PFParams):
        d = asdict(params)
    elif isinstance(params, dict):
        d = asdict(PFParams())
        d.update(params)
        # v6: tvt_window was removed. Ignore it if an older params dict is passed.
        d.pop("tvt_window", None)
    else:
        raise TypeError(f"params must be dict, PFParams, or None. got={type(params)}")

    int_keys = {"n_particles", "n_seeds", "seed0", "hgr_smooth_r", "tw_gr_smooth_r"}
    for k in int_keys:
        d[k] = int(d[k])
    for k in d:
        if k not in int_keys:
            d[k] = float(d[k])

    d["n_particles"] = max(2, int(d["n_particles"]))
    d["n_seeds"] = max(1, int(d["n_seeds"]))
    d["resamp"] = float(np.clip(d["resamp"], 0.01, 0.99))
    d["gr_sig_max"] = max(d["gr_sig_max"], d["gr_sig_min"] + 1e-6)
    d["gr_power"] = float(np.clip(d["gr_power"], 0.5, 4.0))
    d["rate_clip"] = max(0.0, float(d["rate_clip"]))
    d["hgr_smooth_r"] = max(0, int(d["hgr_smooth_r"]))
    d["tw_gr_smooth_r"] = max(0, int(d["tw_gr_smooth_r"]))
    d["jump_prob"] = float(np.clip(d.get("jump_prob", 0.0), 0.0, 1.0))
    d["jump_std"] = max(0.0, float(d.get("jump_std", 10.0)))
    return d


def _pf_param_hash(params):
    d = _pf_params_dict(params)
    s = json.dumps(d, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:10]


def _pf_smooth_1d(x, r=0):
    """PF用の軽い1D平滑化。r<=0なら現行挙動と同じで何もしない。"""
    x = np.asarray(x, dtype=float)
    r = int(r)
    if r <= 0 or len(x) < 3:
        return x.copy()

    win = 2 * r + 1
    if win <= 1:
        return x.copy()

    # pandas rolling meanにして、Windows/Notebookでも依存を増やさず安定させる。
    return (
        pd.Series(x)
        .rolling(window=win, center=True, min_periods=1)
        .mean()
        .to_numpy(dtype=float)
    )


def _pf_gr_sigma(hw, tw_tvt, tw_gr, params):
    """既知TVT区間でtypewell GRとの差からGR sigmaを推定。"""
    kn = hw[hw["TVT_input"].notna() & hw["GR"].notna()]
    if len(kn) < 20:
        gs = params["gr_sig_def"]
    else:
        tw_at_k = np.interp(kn["TVT_input"].to_numpy(float), tw_tvt, tw_gr)
        resid = kn["GR"].to_numpy(float) - tw_at_k
        gs = float(np.nanstd(resid))
        if not np.isfinite(gs) or gs <= 0:
            gs = params["gr_sig_def"]
    gs = np.clip(gs * params["gr_sig_mult"], params["gr_sig_min"], params["gr_sig_max"])
    return float(gs)


def run_particle_filter_param(hw, tw, params=None, seed=0, return_full=True):
    """PF単体でTVTを予測する。

    hwには `TVT_input`, `MD`, `Z`, `GR` が必要。
    train評価では `TVT_input` がNaNの区間を予測し、`TVT` と比較する。
    """
    params = _pf_params_dict(params)

    required_hw = {"TVT_input", "MD", "Z", "GR"}
    required_tw = {"TVT", "GR"}
    if not required_hw.issubset(hw.columns):
        raise ValueError(f"horizontal well missing columns: {sorted(required_hw - set(hw.columns))}")
    if not required_tw.issubset(tw.columns):
        raise ValueError(f"typewell missing columns: {sorted(required_tw - set(tw.columns))}")

    tw_s = tw.sort_values("TVT").dropna(subset=["TVT"]).copy()
    if len(tw_s) == 0:
        raise ValueError("typewell has no valid TVT rows")

    tw_gr_mean = float(tw_s["GR"].mean()) if tw_s["GR"].notna().any() else 0.0
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_gr_mean).to_numpy(float)
    tw_gr = _pf_smooth_1d(tw_gr, params["tw_gr_smooth_r"])

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        full = hw["TVT_input"].to_numpy(float).copy()
        return (full if return_full else np.array([], dtype=float)), 0.0
    if len(kn) == 0:
        raise ValueError("TVT_input has no known rows; PF needs an initial anchor")

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"])
    last_z = float(last["Z"])
    last_md = float(last["MD"])
    # horizontal GRも必要なら平滑化する。r=0なら現行挙動と同じ。
    gr_raw = (
        hw["GR"]
        .astype(float)
        .interpolate(limit_direction="both")
        .fillna(tw_gr_mean)
        .to_numpy(dtype=float)
    )
    hgr_proc = _pf_smooth_1d(gr_raw, params["hgr_smooth_r"])

    # GR sigma推定にも同じ平滑化済みhorizontal GRを使う。
    hw_sigma = hw.copy()
    hw_sigma["GR"] = hgr_proc
    gs = _pf_gr_sigma(hw_sigma, tw_tvt, tw_gr, params)

    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy(float))
    dz = np.diff(tail["Z"].to_numpy(float))
    dm = np.diff(tail["MD"].to_numpy(float))
    m = dm > 0
    init_rate = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N = int(params["n_particles"])
    rng = np.random.default_rng(int(seed))
    last_strat_pos = last_tvt + last_z
    pos = last_strat_pos + params["init_pos_std"] * rng.standard_normal(N)
    rate = init_rate + params["init_rate_std"] * rng.standard_normal(N)
    w = np.ones(N, dtype=float) / N

    md_v = ev["MD"].to_numpy(float)
    z_v = ev["Z"].to_numpy(float)

    full = hw["TVT_input"].to_numpy(float).copy()
    ev_pos = hw.index.get_indexer(ev.index)
    gr_v = hgr_proc[ev_pos]
    pred_ev = np.empty(len(ev), dtype=float)
    prev_md = last_md
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(float(md_v[i] - prev_md), 1.0)
        rate = params["mom"] * rate + params["vn"] * rng.standard_normal(N)
        if params["rate_clip"] > 0:
            rate = np.clip(rate, -params["rate_clip"], params["rate_clip"])
        pos = pos + rate * dm_step + params["pn"] * rng.standard_normal(N)

        # Rare jump particles: keeps high-momentum behavior for most particles,
        # while allowing a small fraction to catch extreme TVT movements.
        if params.get("jump_prob", 0.0) > 0 and params.get("jump_std", 0.0) > 0:
            jump_mask = rng.random(N) < params["jump_prob"]
            if jump_mask.any():
                pos[jump_mask] += params["jump_std"] * rng.standard_normal(int(jump_mask.sum()))

        tvt_p = pos - z_v[i]
        lo = float(tw_tvt[0] - params["tvt_clip_margin"])
        hi = float(tw_tvt[-1] + params["tvt_clip_margin"])
        tvt_p = np.clip(tvt_p, lo, hi)
        pos = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d = np.abs((gr_v[i] - eg) / gs)
        d_pow = np.power(d, params["gr_power"])
        lk = np.exp(-0.5 * np.minimum(d_pow, 600.0))
        lk = np.maximum(lk, 1e-300)

        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = float(w.sum())
        w = w / ws if ws > 0 else np.ones(N, dtype=float) / N

        n_eff = 1.0 / float((w * w).sum())
        if n_eff < params["resamp"] * N:
            cum = np.cumsum(w)
            u0 = rng.uniform(0.0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + params["rp"] * rng.standard_normal(N)
            rate = rate[idx] + params["rr"] * rng.standard_normal(N)
            if params["rate_clip"] > 0:
                rate = np.clip(rate, -params["rate_clip"], params["rate_clip"])
            w = np.ones(N, dtype=float) / N

        pred_ev[i] = float(np.dot(w, pos - z_v[i]))
        prev_md = md_v[i]

    full[ev_pos] = pred_ev
    return (full if return_full else pred_ev), float(log_lik)


def run_pf_lik_ensemble_param(hw, tw, params=None, return_full=True):
    """複数seedのPFをlikelihood重み付き平均する。"""
    params = _pf_params_dict(params)
    preds = []
    liks = []
    seed0 = int(params["seed0"])
    for s in range(int(params["n_seeds"])):
        pred, ll = run_particle_filter_param(hw, tw, params=params, seed=seed0 + s, return_full=return_full)
        preds.append(pred)
        liks.append(ll)

    pred_arr = np.stack(preds, axis=0)
    liks = np.asarray(liks, dtype=float)
    liks_n = liks - np.nanmax(liks)
    weights = np.exp(liks_n / max(float(params["likelihood_scale"]), 1e-6))
    weights = weights / max(float(weights.sum()), 1e-300)
    return (weights[:, None] * pred_arr).sum(axis=0)


def predict_tvt_pf(wid, split="train", params=None):
    """well idを指定してPF予測DataFrameを返す。"""
    hw, tw = load_well(wid, split)
    pred = run_pf_lik_ensemble_param(hw, tw, params=params, return_full=True)

    out = pd.DataFrame({
        "well": wid,
        "row_idx": hw["__row_id"].to_numpy(int),
        "id": [f"{wid}_{int(i)}" for i in hw["__row_id"].to_numpy(int)],
        "MD": hw["MD"].to_numpy(float),
        "Z": hw["Z"].to_numpy(float),
        "GR": hw["GR"].to_numpy(float),
        "TVT_input": hw["TVT_input"].to_numpy(float),
        "pred_tvt": pred,
        "is_eval": hw["TVT_input"].isna().to_numpy(bool),
    })
    if "TVT" in hw.columns:
        out["true_tvt"] = hw["TVT"].to_numpy(float)
        out["error"] = out["pred_tvt"] - out["true_tvt"]
        out["abs_error"] = out["error"].abs()
    return out


In [ ]:
# ===== affine GR再較正(prefix既知区間で GR_hw≈α·GR_tw(TVT)+β をロバスト最小二乗→(GR−β)/α) =====
def recalibrate_gr(hw, tw):
    kn = hw["TVT_input"].notna().to_numpy()
    ttv = tw["TVT"].to_numpy(float); tgr = tw["GR"].to_numpy(float)
    m = np.isfinite(ttv) & np.isfinite(tgr)
    hw2 = hw.copy()
    if m.sum() < 8 or kn.sum() < 100:
        return hw2, (1.0, 0.0)
    o = np.argsort(ttv[m]); tt, tg = ttv[m][o], tgr[m][o]
    gr = hw["GR"].astype(float).interpolate(limit_direction="both").to_numpy()
    x = np.interp(hw["TVT_input"].to_numpy(float)[kn], tt, tg); y = gr[kn]
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 100:
        return hw2, (1.0, 0.0)
    x, y = x[ok], y[ok]; c = np.polyfit(x, y, 1)
    for _ in range(3):
        r = y - np.polyval(c, x); s = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(x, y, 1, w=1.0 / (1.0 + (r / (2 * s)) ** 2))
    a, b = float(c[0]), float(c[1])
    if not np.isfinite(a) or a < 0.3 or a > 3.0:
        return hw2, (1.0, 0.0)
    hw2["GR"] = (gr - b) / a
    return hw2, (a, b)


In [ ]:
# ===== 実行: test各井で r0(raw) と affine を推論 → 2提出 =====
sub = pd.read_csv(DATA / "sample_submission.csv")
sub["well"] = sub["id"].str.rsplit("_", n=1).str[0]
sub["ridx"] = sub["id"].str.rsplit("_", n=1).str[1].astype(int)
pred_raw, pred_aff = {}, {}
t0 = time.time(); wells = sorted(sub["well"].unique())
for i, wid in enumerate(wells):
    hw = pd.read_csv(TEST / f"{wid}__horizontal_well.csv").reset_index(drop=True)
    tw = pd.read_csv(TEST / f"{wid}__typewell.csv").reset_index(drop=True)
    hw["__row_id"] = np.arange(len(hw))
    p_raw = run_pf_lik_ensemble_param(hw, tw, params=R0, return_full=True)
    hw_a, ab = recalibrate_gr(hw, tw)
    p_aff = run_pf_lik_ensemble_param(hw_a, tw, params=R0, return_full=True)
    pred_raw[wid] = np.asarray(p_raw, float); pred_aff[wid] = np.asarray(p_aff, float)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(wells)} ({time.time()-t0:.0f}s)", flush=True)

def fill(pred_map):
    vals = np.zeros(len(sub))
    for j, (wid, ridx) in enumerate(zip(sub["well"].to_numpy(), sub["ridx"].to_numpy())):
        pr = pred_map.get(wid)
        vals[j] = pr[ridx] if (pr is not None and 0 <= ridx < len(pr) and np.isfinite(pr[ridx])) else 0.0
    return vals

v_raw = fill(pred_raw); v_aff = fill(pred_aff)
v_blend = (1 - BLEND_W) * v_raw + BLEND_W * v_aff
# 名前付き2種を保存(比較用)
pd.DataFrame({"id": sub["id"], "tvt": v_raw}).to_csv(OUTDIR / "submission_r0.csv", index=False)
pd.DataFrame({"id": sub["id"], "tvt": v_blend}).to_csv(OUTDIR / "submission_blend.csv", index=False)
# ★Kaggle提出用: submission.csv = SUBMIT で選んだ方
v_sel = v_raw if SUBMIT == "r0" else v_blend
pd.DataFrame({"id": sub["id"], "tvt": v_sel}).to_csv(OUTDIR / "submission.csv", index=False)
print(f"[saved] submission.csv = '{SUBMIT}'  + submission_r0.csv / submission_blend.csv")
print(f"        blend W={BLEND_W}, {len(sub)}行, {time.time()-t0:.0f}s")
print("r0 vs affine 平均絶対差:", float(np.mean(np.abs(v_raw - v_aff))), "ft")
